In [77]:
#WAIT actually im going to use the cosmetic expansion from yesterday to modify instead of starting from scratch 
#deleting all the unnecessary stuff - no more functional group or alkene preservation filters --> these things can be used in post reaction processing if needed


import doranet as dn
from doranet.modules.synthetic.Reaction_Smarts_Forward import op_smarts
from doranet.modules.synthetic.Reaction_Smarts_Retro import op_retro_smarts
from doranet.modules.synthetic.generate_network import (
    get_smiles_from_file,
    Max_Atoms_Filter,
    Ring_Issues_Filter,
    Enol_filter_forward,
    Enol_filter_retro,
    Check_balance_filter,
    Allowed_Elements_Filter,
    Chem_Rxn_dH_Calculator,
    Rxn_dH_Filter,
    Cross_Reaction_Filter,
    Retro_Not_Aromatic_Filter,
)
from datetime import datetime
import time
from rdkit import Chem
from rdkit.Chem import Descriptors
from doranet import metadata, interfaces
import collections.abc
from tal_reaction_whitelist import TAL_REACTION_WHITELIST #THIS IS USED FOR CUSTOM WHITELIST


# ============ CUSTOM FILTERS ============

class Molecular_Weight_Filter(metadata.ReactionFilterBase):
    """Reject reactions if products exceed max molecular weight."""
    
    def __init__(self, max_weight=400):
        self.max_weight = max_weight
    
    def __call__(self, recipe):
        for mol in recipe.products:
            if not isinstance(mol.item, interfaces.MolDatRDKit):
                raise NotImplementedError(
                    f"Filter only works with MolDatRDKit, not {type(mol.item)}"
                )
            
            mw = Descriptors.MolWt(mol.item.rdkitmol)
            
            if mw > self.max_weight:
                return False
        
        return True
    
    @property
    def meta_required(self):
        return interfaces.MetaKeyPacket()


class Minimum_Carbon_Count_Filter(metadata.ReactionFilterBase):
    """Reject molecules with fewer than 8 carbons."""
    
    def __init__(self, min_carbons=0):
        self.min_carbons = min_carbons
    
    def __call__(self, recipe):
        for mol in recipe.products:
            if not isinstance(mol.item, interfaces.MolDatRDKit):
                raise NotImplementedError(
                    f"Filter only works with MolDatRDKit, not {type(mol.item)}"
                )
            
            # Count carbons
            carbon_count = sum(
                1 for atom in mol.item.rdkitmol.GetAtoms() 
                if atom.GetAtomicNum() == 6
            )
            
            if carbon_count < self.min_carbons:
                return False
        
        return True
    
    @property
    def meta_required(self):
        return interfaces.MetaKeyPacket()


# ============ CUSTOM GENERATE FUNCTION ============

def generate_network_tal(
    job_name="default_job",
    starters=False,
    helpers=False,
    gen=2, 
    direction="forward",
    molecule_thermo_calculator=None,
    max_rxn_thermo_change=15,
    max_atoms=None,  # {"C": 50, "O": 8, "N": 2}
    max_molecular_weight=800,  # NEW: Adjustable MW cap
    allow_multiple_reactants="default",
    targets=None,
    strategy="cartesian",  # NEW: "cartesian" or "priority_queue"
    #preserve_delta9=True,  # NEW: Preserve Δ9 alkene
    min_carbons=0,  # NEW: Minimum carbon count
    #enforce_functional_groups=True,  # NEW: Whitelist functional groups
):
    """
    Enhanced generate_network for TAL acid derivatives.
    
    Parameters:
    -----------
    max_atoms : dict
        Atom count limits, e.g., {"C": 50, "O": 8, "N": 2}
    
    max_molecular_weight : float
        Max molecular weight in Daltons (default: 800)
    
    strategy : str
        "cartesian" (blind, exhaustive) or "priority_queue" (targeted)
        If "priority_queue", auto-detects target from targets parameter
    
    min_carbons : int
        Minimum carbon count in molecules (default: 8)
    
    """
    
    if not starters:
        raise Exception("At least one starter is needed to generate a network")

    starters = get_smiles_from_file(starters)
    helpers = get_smiles_from_file(helpers)
    targets = get_smiles_from_file(targets)

    print(f"\n{'='*60}")
    print(f"TAL NETWORK GENERATION")
    print(f"{'='*60}")
    print(f"Job name: {job_name}")
    print(f"Job type: synthetic network expansion {direction}")
    print(f"Strategy: {strategy}")
    if strategy == "priority_queue" and targets:
        print(f"Priority queue targets: {targets}")
    print(f"Atom limits: {max_atoms}")
    print(f"Max molecular weight: {max_molecular_weight} Da")
    #print(f"Preserve Δ9 alkene: {preserve_delta9}")
    print(f"Min carbons: {min_carbons}")
    #print(f"Enforce functional groups: {enforce_functional_groups}")
    print("Job started on:", datetime.now())
    start_time = time.time()

    engine = dn.create_engine()
    network = engine.new_network()

    if helpers:
        for smiles in helpers:
            network.add_mol(engine.mol.rdkit(smiles))

    my_start_i = -1
    for smiles in starters:
        if my_start_i == -1:
            my_start_i = network.add_mol(engine.mol.rdkit(smiles))
        else:
            network.add_mol(engine.mol.rdkit(smiles))

    if direction == "forward":
        smarts_list = [op for op in op_smarts if op.name in TAL_REACTION_WHITELIST]
        #print("SMARTS LIST")
        #for smarts in smarts_list:
           # print(smarts.name)
            
        #TS IS NEW 
    elif direction == "retro":
        smarts_list = op_retro_smarts

    for smarts in smarts_list:
        if smarts.kekulize_flag is False:
            network.add_op(
                engine.op.rdkit(smarts.smarts, drop_errors=True),
                meta={
                    "name": smarts.name,
                    "reactants_stoi": smarts.reactants_stoi,
                    "products_stoi": smarts.products_stoi,
                    "enthalpy_correction": smarts.enthalpy_correction,
                    "ring_issue": smarts.ring_issue,
                    "kekulize_flag": smarts.kekulize_flag,
                    "Retro_Not_Aromatic": smarts.Retro_Not_Aromatic,
                    "number_of_steps": smarts.number_of_steps,
                    "allowed_elements": smarts.allowed_elements,
                    "Reaction_type": smarts.reaction_type,
                    "Reaction_direction": direction,
                },
            )
        if smarts.kekulize_flag is True:
            network.add_op(
                engine.op.rdkit(smarts.smarts, kekulize=True, drop_errors=True),
                meta={
                    "name": smarts.name,
                    "reactants_stoi": smarts.reactants_stoi,
                    "products_stoi": smarts.products_stoi,
                    "enthalpy_correction": smarts.enthalpy_correction,
                    "ring_issue": smarts.ring_issue,
                    "kekulize_flag": smarts.kekulize_flag,
                    "Retro_Not_Aromatic": smarts.Retro_Not_Aromatic,
                    "number_of_steps": smarts.number_of_steps,
                    "allowed_elements": smarts.allowed_elements,
                    "Reaction_type": smarts.reaction_type,
                    "Reaction_direction": direction,
                },
            )

    print(f"Number of operators added to network: {len(network.ops)}")

    # ====== CHOOSE STRATEGY ======
    if strategy == "cartesian":
        strat = engine.strat.cartesian(network)
    elif strategy == "priority_queue":
        if not targets:
            raise ValueError(
                "priority_queue strategy requires targets parameter to be set"
            )
        # Use first target as the ranker target
        target_for_ranker = targets[0] if isinstance(targets, list) else targets
        ranker = engine.ranker.smiles_distance(target_for_ranker)
        strat = engine.strat.priority_queue(network, ranker=ranker)
        print(f"Using priority queue strategy, targeting: {target_for_ranker}")
    else:
        raise ValueError(
            f"Unknown strategy: {strategy}. Use 'cartesian' or 'priority_queue'"
        )
    # =============================

    periodic_table = Chem.GetPeriodicTable()

    if max_atoms is None:
        max_atoms_dict_num = None
    else:
        max_atoms_dict_num = dict()
        for key in max_atoms:
            max_atoms_dict_num[periodic_table.GetAtomicNumber(key)] = max_atoms[key]

    # Build reaction plan with custom filters
    if direction == "forward":
        reaction_plan = (
            Max_Atoms_Filter(max_atoms_dict_num)
            >> Molecular_Weight_Filter(max_molecular_weight)
            >> Ring_Issues_Filter()
            >> Enol_filter_forward()
            >> Check_balance_filter()
            >> Allowed_Elements_Filter()
        )
        
        # Add optional filters
       #if preserve_delta9:
          #  reaction_plan = reaction_plan >> Preserve_Delta9_Alkene_Filter()
        
        if min_carbons > 0:
            reaction_plan = reaction_plan >> Minimum_Carbon_Count_Filter(min_carbons)
        
        #  enforce_functional_groups:
            #reaction_plan = reaction_plan >> Functional_Group_Whitelist_Filter()
        
        # Add thermodynamics filters at the end
        reaction_plan = (
            reaction_plan
            >> Chem_Rxn_dH_Calculator("dH", "forward", molecule_thermo_calculator)
            >> Rxn_dH_Filter(max_rxn_thermo_change, "dH")
        )
        #HOLD UP - is this from original or is this something i added - I DO NOT REMEMBER ADDING THIS
        #Should thermodynamics be filter or analysis --> actually im 80% sure this is from original so i wont touch
        
        recipe_filter = None

    elif direction == "retro":
        reaction_plan = (
            Max_Atoms_Filter(max_atoms_dict_num)
            >> Molecular_Weight_Filter(max_molecular_weight)
            >> Ring_Issues_Filter()
            >> Retro_Not_Aromatic_Filter()
            >> Enol_filter_retro()
            >> Allowed_Elements_Filter()
            >> Check_balance_filter()
        )
        
        # Add optional filters
        #if preserve_delta9:
         #   reaction_plan = reaction_plan >> Preserve_Delta9_Alkene_Filter()"""
        
        if min_carbons > 0:
            reaction_plan = reaction_plan >> Minimum_Carbon_Count_Filter(min_carbons)
        
        #if enforce_functional_groups:
         #   reaction_plan = reaction_plan >> Functional_Group_Whitelist_Filter()"""
        
        # Add thermodynamics filters at the end
        reaction_plan = (
            reaction_plan
            >> Chem_Rxn_dH_Calculator("dH", "retro", molecule_thermo_calculator)
            >> Rxn_dH_Filter(max_rxn_thermo_change, "dH")
        )
        
        recipe_filter = Cross_Reaction_Filter(tuple(range(my_start_i)))

    if allow_multiple_reactants != "default":
        if allow_multiple_reactants is True:
            recipe_filter = None
        elif allow_multiple_reactants is False:
            recipe_filter = Cross_Reaction_Filter(tuple(range(my_start_i)))

    bundle_filter = engine.filter.bundle.coreactants(tuple(range(my_start_i)))

    ini_number = len(network.mols)

    strat.expand(
        num_iter=gen,
        reaction_plan=reaction_plan,
        bundle_filter=bundle_filter,
        recipe_filter=recipe_filter,
        save_unreactive=False,
    )

    if targets is not None:
        print("\nChecking for targets...")
        to_check = set()
        if isinstance(targets, str):
            to_check.add(Chem.MolToSmiles(Chem.MolFromSmiles(targets)))
        else:
            for i in targets:
                to_check.add(Chem.MolToSmiles(Chem.MolFromSmiles(i)))

        targets_found = []
        for mol in network.mols:
            if (
                network.reactivity[network.mols.i(mol.uid)] is True
                and mol.uid in to_check
            ):
                targets_found.append(mol.uid)
                print(f"  ✓ Target found: {mol.uid}")
        
        if len(targets_found) == 0:
            print(f"  ✗ No targets found")

    print("\n" + "="*60)
    print("NETWORK GENERATION COMPLETE")
    print("="*60)
    print(f"Number of generations: {gen}")
    print(f"Number of operators: {len(network.ops)}")
    print(f"Number of molecules before expansion: {ini_number}")
    print(f"Number of molecules after expansion: {len(network.mols)}")
    print(f"Number of reactions: {len(network.rxns)}")

    end_time = time.time()
    elapsed_time = (end_time - start_time) / 60
    print(f"Time used: {elapsed_time:.2f} minutes")
    print("="*60 + "\n")

    network.save_to_file(f"{job_name}_{direction}_saved_network")

    return network

In [78]:

# Use cell
network = generate_network_tal(
  starters=["CC1=CC(O)=CC(=O)O1"],
    helpers=["[H][H]", "O", "CCO"],
    max_atoms={
        "C": 25,
        "O": 4,
        "N": 2,
        "S": 0,
        "P": 0,
        "F": 0,
        "Cl": 0,
        "Br": 0,
        "I": 0,
    },
    job_name="tal_basic",
    gen=3,
    direction="forward",
    min_carbons = 0,
    
)

# number of molecules and rxns 
print(f"Generated {len(network.mols)} molecules")
print(f"Generated {len(network.rxns)} reactions")

# Print all molecules
for i, mol in enumerate(network.mols):
    print(f"Molecule {i}: {mol.uid}")  # uid is the SMILES string


#WAIT MAJOR ISSUE - all the things i thought i was passing in to the generation network are actually being hardcoded in. 



TAL NETWORK GENERATION
Job name: tal_basic
Job type: synthetic network expansion forward
Strategy: cartesian
Atom limits: {'C': 25, 'O': 4, 'N': 2, 'S': 0, 'P': 0, 'F': 0, 'Cl': 0, 'Br': 0, 'I': 0}
Max molecular weight: 800 Da
Min carbons: 0
Job started on: 2026-06-11 04:31:38.727961
Number of operators added to network: 152


[04:31:39] reactant 1 has no mapped atoms.
[04:31:39] reactant 1 has no mapped atoms.
[04:31:40] reactant 1 has no mapped atoms.
[04:31:40] reactant 1 has no mapped atoms.
[04:31:40] reactant 1 has no mapped atoms.
[04:31:40] reactant 1 has no mapped atoms.
[04:31:40] reactant 2 has no mapped atoms.



NETWORK GENERATION COMPLETE
Number of generations: 3
Number of operators: 152
Number of molecules before expansion: 4
Number of molecules after expansion: 123
Number of reactions: 149
Time used: 0.18 minutes

Generated 123 molecules
Generated 149 reactions
Molecule 0: [H][H]
Molecule 1: O
Molecule 2: CCO
Molecule 3: Cc1cc(O)cc(=O)o1
Molecule 4: Cc1cccc(=O)o1
Molecule 5: CCOc1cc(C)oc(=O)c1
Molecule 6: CCc1c(O)cc(=O)oc1C
Molecule 7: CCc1c(O)cc(C)oc1=O
Molecule 8: CCc1cc(C)oc(=O)c1
Molecule 9: CCc1ccc(=O)oc1C
Molecule 10: CCc1ccc(C)oc1=O
Molecule 11: CC1=CC2C3C=CC(C)(OC3=O)C2C(=O)O1
Molecule 12: CC12C=CC(C(=O)O1)C1C=CC(=O)OC12C
Molecule 13: CC1=CC2C(C(=O)O1)C1C=CC2(C)OC1=O
Molecule 14: CC12C=CC(C(=O)O1)C1(C)OC(=O)C=CC21
Molecule 15: CC
Molecule 16: CCOc1cc(=O)oc(C)c1CC
Molecule 17: CCOc1cc(C)oc(=O)c1CC
Molecule 18: CCc1c(C)oc(=O)c(CC)c1O
Molecule 19: CCc1cc(=O)oc(C)c1CC
Molecule 20: CCc1cc(C)oc(=O)c1CC
Molecule 21: CCC12C=C(C)OC(=O)C1C1(C)C=CC2C(=O)O1
Molecule 22: CCC1=CC(=O)OC2(C)C1C1C=

In [79]:
pretreat_networks(
    networks=[network],
    starters=[STARTER],
    helpers=["[H][H]", "O", "CCO"],  
    total_generations=3,
    job_name="TAL"
)

helpers_to_pass = ["[H][H]", "O", "CCO"]
print("helpers being passed:", helpers_to_pass)

pretreat_networks(
    networks=[network],
    starters=[STARTER],
    helpers=helpers_to_pass,
    total_generations=3,
    job_name="TAL"
)


import json
pretreated = json.load(open("TAL_network_pretreated.json"))
print(f"Reactions in pretreated network: {len(pretreated)}")

target = "CCc1ccc(C)oc1=O"
from rdkit import Chem
target_canonical = Chem.MolToSmiles(Chem.MolFromSmiles(target))

# verify target is reachable
all_products = set()
for rxn in pretreated:
    for p in rxn.split(">")[3].split("."):
        all_products.add(p)
print(f"Target in pretreated network products: {target_canonical in all_products}")

pathway_finder(
    starters=[STARTER],
    helpers=["[H][H]", "O", "CCO"],
    target=[target_canonical],
    search_depth=3,
    max_num_rxns=20,
    job_name="TAL"
)

Job name: TAL
Job type: post-processing pretreat networks
Job started on: 2026-06-11 04:31:54.488292
Loading networks, it may take a while if loading large networks from file
Loading network 1 from memory
Number of reations in network 1: 149
Networks loaded, now generating reaction strings
Reaction strings generation finished
Removing unconnected reactions
Unconnected reactions removed
Total number of reactions after pretreatment: 140
Time used for network pretreatment: 0.00 minutes

helpers being passed: ['[H][H]', 'O', 'CCO']
Job name: TAL
Job type: post-processing pretreat networks
Job started on: 2026-06-11 04:31:54.560012
Loading networks, it may take a while if loading large networks from file
Loading network 1 from memory
Number of reations in network 1: 149
Networks loaded, now generating reaction strings
Reaction strings generation finished
Removing unconnected reactions
Unconnected reactions removed
Total number of reactions after pretreatment: 140
Time used for network pretr

In [82]:
# import os
# print("Working directory:", os.getcwd())

# with open("TAL_pathways.txt") as f:
#     print(f.read())


# intermediate viability check

from rdkit import Chem
from rdkit.Chem import Descriptors

def assess_intermediate_viability(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"viable": False, "issues": ["invalid SMILES"]}
    
    issues = []
    
    # Very high MW
    mw = Descriptors.MolWt(mol)
    if mw > 500:
        issues.append(f"high MW ({mw:.0f} Da)")
    
    # Strained rings
    for ring in mol.GetRingInfo().AtomRings():
        if len(ring) < 4:
            issues.append(f"strained {len(ring)}-membered ring")
    
    # Radical centers
    for atom in mol.GetAtoms():
        if atom.GetNumRadicalElectrons() > 0:
            issues.append("radical center")
    
    # Charged species
    if sum(abs(atom.GetFormalCharge()) for atom in mol.GetAtoms()) > 0:
        issues.append("charged species")
    
    # Peroxide
    if mol.HasSubstructMatch(Chem.MolFromSmarts("[OX2][OX2]")):
        issues.append("peroxide group")

    return {
        "smiles": smiles,
        "viable": len(issues) == 0,
        "issues": issues
    }


# --- Parse pathways file to get reaction names and their intermediates ---
with open("TAL_pathways.txt") as f:
    raw = f.read()

HELPERS = {"CCO", "O", "[H][H]"}

cool_reactions = {}
current_smiles = []
current_names = []
in_smiles_block = False
in_names_block = False

blocks = raw.strip().split("pathway number ")
for block in blocks[1:]:  # skip empty first split
    lines = [l.strip() for l in block.split("\n") if l.strip()]
    
    smiles_lines = []
    name_lines = []
    found_smiles_header = False
    
    for line in lines:
        if line.startswith("reaction SMILES, name"):
            found_smiles_header = True
            continue
        if found_smiles_header:
            if ">>" in line:
                smiles_lines.append(line)
            elif line.startswith("No_Thermo") or line[0].isdigit():
                break
            elif smiles_lines:  # names come after smiles
                name_lines.append(line)
    
    # pair each reaction name with its intermediate viability
    for i, name in enumerate(name_lines):
        if i >= len(smiles_lines):
            break
        rxn = smiles_lines[i]
        reas, pros = rxn.split(">>")
        all_mols = reas.split(".") + pros.split(".")
        intermediates = [
            s for s in all_mols
            if s not in HELPERS
            and s != STARTER
            and s != target_canonical
        ]
        all_viable = all(assess_intermediate_viability(s)["viable"] for s in intermediates)
        cool_reactions[name] = 1 if all_viable else 0

print("Coolness scores assigned to reactions:")
for name, score in cool_reactions.items():
    status = "✓" if score == 1 else "✗"
    print(f"  {status} {name}")

Coolness scores assigned to reactions:
  ✓ Ether Synthesis by Dehydration
  ✓ Electrophilic Aromatic Alkylation with Alcohols
  ✓ Hydrogenolysis of Ethers
  ✓ Hydrodeoxygenation of Alcohol, Classic Synthesis of Aldehydes from Carboxylic Acids


In [85]:
from doranet.modules.post_processing.post_processing import pathway_ranking

pathway_ranking(
    starters=[STARTER],
    helpers=["[H][H]", "O", "CCO"],
    target=[target_canonical],
    job_name="TAL",
    weights= {
        "reaction_thermo": 0,
        "number_of_steps": 4,
        "by_product_number": 2,
        "atom_economy": 1,
        "salt_score": 0,
        "in_reaxys": 0,
        "coolness": 2,
    },
    cool_reactions=cool_reactions
)

with open("TAL_ranked_pathways.txt") as f:
    print(f.read())

Job name: TAL
Job type: pathway ranking
Job started on: 2026-06-11 04:49:49.953423
Start ranking pathways. Number of pathways after sanitization: 5
Atom economy ranking working
Min_eco 0.627271695457744
Max_eco 0.7931640221359847
None_eco 0
Atom_economy ranking finished
Number of steps ranking working
Number of steps ranking finished
By product_number ranking working
You can adjust multi-processing number to speed it up
By product_number ranking finished
Coolness ranking working
Coolness ranking finished
min score 2.0
max score 7.0

Pathway ranking finished. Pathway scores:

ranking 1
final score 7.0
Max reaction enthalpy score 0  x  0  =  0
Number of reactions score 1.0  x  4  =  4.0
By-product score 1.0  x  2  =  2.0
Pathway atom economy score 1.0  x  1  =  1.0
Salt score 0  x  0  =  0
Reaxys score 0  x  0  =  0
Cool score 0.0  x  2  =  0.0
atom economy 0.7931640221359847
pathway by-product 53
intermediate by-product {'CCc1ccc(C)oc1=O': 23, 'Cc1cc(O)cc(=O)o1': 26, 'Cc1cccc(=O)o1': 23

In [72]:
from doranet.modules.post_processing.post_processing import pathway_visualization

pathway_visualization(
    starters=[STARTER],
    helpers=["[H][H]", "O", "CCO"],
    job_name="TAL",
    reaxys_result_name=None
)

Job name: TAL
Job type: pathway visualization
Job started on: 2026-06-11 03:22:19.161014
pygraphviz is NOT installed.
Graphviz is NOT installed.
A custom node layout will be used for pathway visualization


[03:22:19] WARNING: not removing hydrogen atom without neighbors
[03:22:19] WARNING: not removing hydrogen atom without neighbors
[03:22:19] WARNING: not removing hydrogen atom without neighbors


Number of pathways:  5
Number of reactions in reaxys:  0
Working on creating pages
You can adjust multi-processing number to speed up PDF generation
Finished with pages, writing to pdf
Time used for pathway visualization: 0.28 minutes

A custom node layout was used for pathway visualization
For a better layout, please install pygraphviz and Graphviz with the following command:
conda install conda-forge::pygraphviz
which should install both packages together



In [86]:
import os
import subprocess

pdf_path = os.path.join(os.getcwd(), "TAL_pathways_visualized.pdf")
print("PDF location:", pdf_path)
os.startfile(pdf_path)  # opens with default PDF viewer on Windows

PDF location: C:\Users\ashvi\LemniscaInternship\TAL Expansion\TAL_pathways_visualized.pdf
